# Build a Transformer

## Initial Setup

In [1]:
# Reload modules when they are edited
%load_ext autoreload
%autoreload 2

In [2]:
# Initial imports
import os
import sys
import spacy
import requests
import tarfile
import torch

import numpy as np

from torch import Tensor
from typing import List, Tuple, Dict
from pprint import pprint
from pathlib import Path
from collections import Counter

In [3]:
# Add project path to sys.path for module imports
project_path: str = str(
    Path(os.getcwd()) / "learning" / "build_a_text_to_image_generator_from_scratch"
)
if project_path not in sys.path:
    sys.path.append(project_path)
    print(f">>> Added {project_path} to sys.path")

>>> Added /llm_app/learning/build_a_text_to_image_generator_from_scratch to sys.path


In [4]:
# Load custom modules
from utils.transformer import (
    Batch,
    attention,
    MultiHeadedAttention,
    create_model,
    NoamOpt,
    LabelSmoothing,
    SimpleLossCompute,
    de2en,
    DEVICE,
)

## Word Embedding and Positional Encoder

### Word Tokenization with the Spacy Library

In [5]:
url: str = (
    "https://raw.githubusercontent.com/neychev/"
    "small_DL_repo/master/datasets/Multi30k/training.tar.gz"
)

# Create the data folder if it doesn't exist
DATA_FOLDER: str = "/llm_app/learning/build_a_text_to_image_generator_from_scratch/files"
os.makedirs(DATA_FOLDER, exist_ok=True)

# Download the training data if it doesn't exist
if not os.path.exists(f"{DATA_FOLDER}/training.tar.gz"):

    # Download the file
    fb1 = requests.get(url)
    with open(f"{DATA_FOLDER}/training.tar.gz", "wb") as f:
        f.write(fb1.content)


train = tarfile.open(f"{DATA_FOLDER}/training.tar.gz")
train.extractall(DATA_FOLDER)

train.close()

In [6]:
# Disk usage of the downloaded data
!du -sh {DATA_FOLDER}/*

93M	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/de2en.pth
86M	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/de2en.zip
2.1M	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/train.de
1.8M	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/train.en
1.2M	/llm_app/learning/build_a_text_to_image_generator_from_scratch/files/training.tar.gz


In [7]:
with open(str(Path(DATA_FOLDER) / "train.de"), "rb") as fb:
    trainde = fb.readlines()

with open(str(Path(DATA_FOLDER) / "train.en"), "rb") as fb:
    trainen = fb.readlines()

trainde: List[str] = [i.decode("utf-8").strip() for i in trainde]
trainen: List[str] = [i.decode("utf-8").strip() for i in trainen]

In [8]:
print(f">>> The length of the list trainde is {len(trainde)}")
print(f">>> The length of the list trainen is {len(trainen)}")

print()
print(">>> The first five elements of the list trainde are")
pprint(trainde[-5:])

print()
print(">>> The first five elements of the list trainen are")
pprint(trainen[-5:])

>>> The length of the list trainde is 29001
>>> The length of the list trainen is 29001

>>> The first five elements of the list trainde are
['Ein Bergsteiger übt an einer Kletterwand.',
 'Zwei Bauarbeiter arbeiten auf einer Straße vor einem Hauses.',
 'Ein älterer Mann sitzt mit einem Jungen mit einem Wagen vor einer Fassade.',
 'Ein Mann in Shorts und Hawaiihemd lehnt sich über das Geländer eines '
 'Lotsenboots, mit Nebel und Bergen im Hintergrund.',
 '']

>>> The first five elements of the list trainen are
['A rock climber practices on a rock climbing wall.',
 "Two male construction workers are working on a street outside someone's home",
 'An elderly man sits outside a storefront accompanied by a young boy with a '
 'cart.',
 'A man in shorts and a Hawaiian shirt leans over the rail of a pilot boat, '
 'with fog and mountains in the background.',
 '']


In [9]:
try:
    de_tokenizer = spacy.load("de_core_news_sm")
except IOError:
    os.system("python -m spacy download de_core_news_sm")
    de_tokenizer = spacy.load("de_core_news_sm")

try:
    en_tokenizer = spacy.load("en_core_web_sm")
except IOError:
    os.system("python -m spacy download en_core_web_sm")
    en_tokenizer = spacy.load("en_core_web_sm")

In [10]:
tokenized_de: List[str] = [tok.text for tok in de_tokenizer.tokenizer(trainde[0])]
tokenized_en: List[str] = [tok.text for tok in en_tokenizer.tokenizer(trainen[0])]

print(tokenized_de)
print(tokenized_en)

['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [11]:
tokenized_de1: List[str] = [tok.text for tok in de_tokenizer.tokenizer(trainde[1])]
tokenized_en1: List[str] = [tok.text for tok in en_tokenizer.tokenizer(trainen[1])]

print(tokenized_de1)
print(tokenized_en1)

['Mehrere', 'Männer', 'mit', 'Schutzhelmen', 'bedienen', 'ein', 'Antriebsradsystem', '.']
['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']


In [12]:
PAD: int = 0
UNK: int = 1

In [13]:
en_tokens: List[List[str]] = [
    ["BOS"] + [tok.text for tok in en_tokenizer.tokenizer(x)] + ["EOS"] for x in trainen
]

word_count = Counter()
for sentence in en_tokens:
    for word in sentence:
        word_count[word] += 1

frequency: List[Tuple[str, int]] = word_count.most_common(50000)
total_en_words: int = len(frequency) + 2

# A dictionary mapping tokens to indexes
en_word_dict = {w[0]: idx + 2 for idx, w in enumerate(frequency)}
en_word_dict["PAD"] = PAD
en_word_dict["UNK"] = UNK

# Another dictionary to map indexes to tokens
en_idx_dict = {v: k for k, v in en_word_dict.items()}

In [14]:
enidx: List[int] = [en_word_dict.get(i, UNK) for i in tokenized_en]

print(enidx)

[19, 25, 15, 1165, 804, 17, 57, 84, 334, 1329, 5]


In [15]:
entokens: List[str] = [en_idx_dict.get(i, "UNK") for i in enidx]
print(entokens)

en_phrase: str = " ".join(entokens)
for x in """?:;.,'("-!&)%""":
    en_phrase = en_phrase.replace(f" {x}", f"{x}")

print(en_phrase)

['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
Two young, White males are outside near many bushes.


In [16]:
# Do the same for German phrases
de_tokens: List[List[str]] = [
    ["BOS"] + [tok.text for tok in de_tokenizer.tokenizer(x)] + ["EOS"] for x in trainde
]

de_word_count = Counter()
for sentence in de_tokens:
    for word in sentence:
        de_word_count[word] += 1

defrequency: List[Tuple[str, int]] = de_word_count.most_common(50000)
total_de_words: int = len(defrequency) + 2

de_word_dict: Dict[str, int] = {w[0]: idx + 2 for idx, w in enumerate(defrequency)}
de_word_dict["PAD"] = PAD
de_word_dict["UNK"] = UNK

de_idx_dict: Dict[int, str] = {v: k for k, v in de_word_dict.items()}

In [17]:
deidx: List[int] = [de_word_dict.get(i, UNK) for i in tokenized_de]
print(deidx)

[21, 85, 257, 31, 87, 22, 94, 7, 16, 112, 5497, 3161, 4]


In [18]:
detokens: List[str] = [de_idx_dict.get(i, "UNK") for i in deidx]
print(detokens)

de_phrase: str = " ".join(detokens)

for x in """?:;.,'("-!&)%""":
    de_phrase: str = de_phrase.replace(f" {x}", f"{x}")

print(de_phrase)

['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


### A Sequence Padding Function

In [19]:
out_en_ids: List[List[int]] = [[en_word_dict.get(w, UNK) for w in s] for s in en_tokens]
out_de_ids: List[List[int]] = [[de_word_dict.get(w, UNK) for w in s] for s in de_tokens]

# Sorted observations in the training dataset by the length of the German
# phrases before placing them into batches. This method ensures that the observations
# within each batch are of a comparable length, consequently decreasing the need for
# padding. As a result, this approach not only reduces the overall size of the training
# data but also accelerates the training process.
sorted_ids: List[int] = sorted(range(len(out_de_ids)), key=lambda x: len(out_de_ids[x]))

out_de_ids: List[List[int]] = [out_de_ids[x] for x in sorted_ids]
out_en_ids: List[List[int]] = [out_en_ids[x] for x in sorted_ids]

In [20]:
batch_size: int = 128

idx_list: np.ndarray = np.arange(0, len(de_tokens), batch_size)
np.random.shuffle(idx_list)

batch_indexs: List[np.ndarray] = []
for idx in idx_list:
    batch_indexs.append(np.arange(idx, min(len(de_tokens), idx + batch_size)))

In [21]:
def seq_padding(X: List[List[int]], padding: int = PAD) -> np.ndarray:
    """
    Pad sequences to the same length.

    Parameters
    ----------
    X : List[List[int]]
        A list of sequences, where each sequence is a list of integers.
    padding : int, optional
        The value to use for padding, by default PAD (which is 0).

    Returns
    -------
    np.ndarray
        A 2D numpy array where each sequence has been padded to the same length.
    """

    L: List[int] = [len(x) for x in X]
    ML: int = max(L)

    padded_seq: np.ndarray = np.array(
        [np.concatenate([x, [padding] * (ML - len(x))]) if len(x) < ML else x for x in X]
    )

    return padded_seq

In [22]:
# Each batch has 128 pairs of numerical representations of German and English phrases, respectively
batches: List[Batch] = []

for b in batch_indexs:

    batch_en: List[List[int]] = [out_en_ids[x] for x in b]
    batch_en: np.ndarray = seq_padding(X=batch_en, padding=PAD)
    batch_de: List[List[int]] = [out_de_ids[x] for x in b]
    batch_de: np.ndarray = seq_padding(X=batch_de, padding=PAD)

    batches.append(Batch(src=batch_de, tgt=batch_en))

### Input Embedding from Word Embedding and Positional Encoding

In [23]:
src_vocab: int = len(de_word_dict)
tgt_vocab: int = len(en_word_dict)

print(f">>> There are {src_vocab} distinct German tokens")
print(f">>> There are {tgt_vocab} distinct English tokens")

>>> There are 19214 distinct German tokens
>>> There are 10837 distinct English tokens


## Creating an Encoder–Decoder Transformer

### Coding the Attention Mechanism

In [24]:
# Test 'attention' function
query: Tensor = torch.rand(2, 4, 256)  # (batch_size, seq_len, d_k)
key: Tensor = torch.rand(2, 4, 256)  # (batch_size, seq_len, d_k)
value: Tensor = torch.rand(2, 4, 256)  # (batch_size, seq_len, d_v)

context, p_attn = attention(query=query, key=key, value=value)

print(f">>> The shape of the 'context' tensor is {context.shape}")
print(f">>> The shape of the 'p_attn' tensor is {p_attn.shape}")

>>> The shape of the 'context' tensor is torch.Size([2, 4, 256])
>>> The shape of the 'p_attn' tensor is torch.Size([2, 4, 4])


In [25]:
# Sum the attention weights across the last dimension to verify that they sum to 1
p_attn[0].sum(axis=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000])

In [26]:
# Test 'MultiHeadedAttention' class
multi_head_attn = MultiHeadedAttention(h=8, d_model=256)

output: Tensor = multi_head_attn(query=query, key=key, value=value)

print(f">>> The shape of the output tensor from 'MultiHeadedAttention' is {output.shape}")

>>> The shape of the output tensor from 'MultiHeadedAttention' is torch.Size([2, 4, 256])


### Defining the Transformer Class

### Creating a language translator

In [27]:
# Instantiate the full transformer model using the 'create_model' function
model = create_model(
    src_vocab=src_vocab,
    tgt_vocab=tgt_vocab,
    N=6,
    d_model=256,
    d_ff=1024,
    h=8,
    dropout=0.1,
)

## Training and using the German-to-English Translator

### Training the Encoder–Decoder Transformer

In [36]:
MODEL_FILE_PATH: str = os.path.join(DATA_FOLDER, "de2en.pth")

print(f">>> The model has been created (or will be saved) at:\n\t* {MODEL_FILE_PATH} ...")

>>> The model has been created (or will be saved) at:
	* /llm_app/learning/build_a_text_to_image_generator_from_scratch/files/de2en.pth ...


In [29]:
# Check if a pre-trained model exists at the specified path.
# If it does, load the model's state dictionary from the file.
# If not, prepare the model for training from scratch by setting up the optimizer and loss function
if os.path.exists(MODEL_FILE_PATH):

    print(f">>> The model will be loaded from {MODEL_FILE_PATH}...")
    state_dict: Dict[str, Tensor] = torch.load(
        f=MODEL_FILE_PATH,
        map_location=torch.device("cpu"),
        weights_only=True,
    )
    model.load_state_dict(state_dict)

else:
    print(
        f">>> No pre-trained model found at {MODEL_FILE_PATH}. "
        f"The model will be trained from scratch."
    )

    optimizer = NoamOpt(
        model_size=256,
        factor=1,
        warmup=2000,
        optimizer=torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9),
    )

    criterion = LabelSmoothing(size=tgt_vocab, padding_idx=PAD, smoothing=0.0)

    loss_func = SimpleLossCompute(
        generator=model.generator, criterion=criterion, opt=optimizer
    )

>>> The model will be loaded from /llm_app/learning/build_a_text_to_image_generator_from_scratch/files/de2en.pth...


In [30]:
# Train from scratch if no pre-trained model is found at the specified path
if not os.path.exists(MODEL_FILE_PATH):

    # Train for few epochs
    for epoch in range(5):

        # Put the model in training mode to enable dropout and other training-specific behaviors
        model.train()

        tloss: float = 0.0
        tokens: int = 0

        # Loop over the batches and compute the loss for each batch, accumulating the total loss and number of tokens for logging
        for batch in batches:

            out: torch.Tensor = model(
                src=batch.src,
                tgt=batch.tgt,
                src_mask=batch.src_mask,
                tgt_mask=batch.tgt_mask,
            )

            loss: torch.Tensor = loss_func(x=out, y=batch.tgt_y, norm=batch.ntokens)

            tloss += loss.item()
            tokens += batch.ntokens

        print(f"Epoch {epoch}, average loss: {tloss/tokens}")

    torch.save(model.state_dict(), MODEL_FILE_PATH)

In [31]:
# Show the size of the saved model file
!ls -lh {MODEL_FILE_PATH}

-rw-rw-r-- 1 worker-user worker-user 93M Jul 27  2024 /llm_app/learning/build_a_text_to_image_generator_from_scratch/files/de2en.pth


### Translating German to English with the Trained Model

In [32]:
model.load_state_dict(torch.load(MODEL_FILE_PATH, weights_only=True, map_location=DEVICE))

<All keys matched successfully>

In [33]:
model.eval();

In [34]:
for i in range(5):
    print(f">>> Original German:\n\t* {trainde[100+i]}")
    print(f">>> Original English:\n\t* {trainen[100+i]}")

    translation: str = de2en(
        ger=trainde[100 + i],
        model=model,
        de_tokenizer=de_tokenizer,
        de_word_dict=de_word_dict,
        en_word_dict=en_word_dict,
        en_idx_dict=en_idx_dict,
    )

    print(f">>> Translated English:\n\t* {translation}")
    print()

>>> Original German:
	* Männliches Kleinkind in einem roten Hut, das sich an einem Geländer festhält.
>>> Original English:
	* Toddler boy in a red hat holding on to some railings.
>>> Translated English:
	* Toddler boy in a red hat holding on to some railings.

>>> Original German:
	* Drei Hunde stehen auf einer Wiese und eine Person kniet in der Nähe.
>>> Original English:
	* Three dogs stand in a grassy field while a person kneels nearby.
>>> Translated English:
	* Three dogs stand in a grassy field and one person near them.

>>> Original German:
	* Ein Mann steht vor einem Hochhaus.
>>> Original English:
	* A man is standing in front of a skyscraper
>>> Translated English:
	* A man is standing in front of a skyscraper

>>> Original German:
	* Eine Frau fährt ihr Baby in einem Sportwagen im örtlichen Park spazieren.
>>> Original English:
	* A woman is walking her baby with a stroller at the local park.
>>> Translated English:
	* A woman is walking her baby in a stroller at the local

#### Exercise 2.5

In [35]:
for i in range(10, 12):
    print(f">>> Original German:\n\t* {trainde[i]}")
    print(f">>> Original English:\n\t* {trainen[i]}")

    translation: str = de2en(
        ger=trainde[i],
        model=model,
        de_tokenizer=de_tokenizer,
        de_word_dict=de_word_dict,
        en_word_dict=en_word_dict,
        en_idx_dict=en_idx_dict,
    )

    print(f">>> Translated English:\n\t* {translation}")
    print()

>>> Original German:
	* Eine Ballettklasse mit fünf Mädchen, die nacheinander springen.
>>> Original English:
	* A ballet class of five girls jumping in sequence.
>>> Translated English:
	* A ballet class of five girls jumping in sequence.

>>> Original German:
	* Vier Typen, von denen drei Hüte tragen und einer nicht, springen oben in einem Treppenhaus.
>>> Original English:
	* Four guys three wearing hats one not are jumping at the top of a staircase.
>>> Translated English:
	* Four guys, three wearing hats, one not, are leaping on top of a staircase.

